In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from arch import arch_model
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("success")

In [ ]:
# 2. Download Price Data (Explicit Column Mapping)

# Define ticker dictionary (friendly name -> Yahoo symbol)
tickers = {
    'S&P 500': '^GSPC',
    'EURO STOXX 50': '^STOXX50E',
    'Nikkei 225': '^N225',
    'FTSE 100': '^FTSE'
}

print("Downloading data from Yahoo Finance...")

# Download all tickers
raw = yf.download(
    list(tickers.values()),
    start='2000-01-01',
    end='2026-09-03',
    group_by='ticker',
    auto_adjust=False,
    progress=True
)

# --- FIX: Explicitly assign each column by its ticker symbol ---
# This guarantees the correct mapping, regardless of column order.
price_data = pd.DataFrame()
for friendly_name, symbol in tickers.items():
    price_data[friendly_name] = raw[symbol]['Adj Close']

# Drop rows where any index is missing data (aligns all indices to common dates)
price_data = price_data.dropna()

# --- Summary and Preview (3 earliest + 3 latest) ---
print(f"\n✅ Data downloaded successfully!")
print(f"   {price_data.shape[0]} trading days")
print(f"   From: {price_data.index[0].date()}")
print(f"   To:   {price_data.index[-1].date()}")

print("\n📅 Earliest 3 trading days:")
display(price_data.head(3))

print("\n📅 Latest 3 trading days (most current):")
display(price_data.tail(3))

In [ ]:
# 3.Log Returns (Scaled by 100)
# Log return formula: r_t = 100 * ln(P_t / P_{t-1})
returns = 100 * np.log(price_data / price_data.shift(1)).dropna()

print(f"Returns matrix: {returns.shape[0]} days, {returns.shape[1]} indices\n")

print("Descriptive Statistics (Daily % Returns):")
display(returns.describe().T)

print("\nFirst 3 rows of returns:")
display(returns.head(3))

print("\nLast 3 rows of returns:")
display(returns.tail(3))

In [ ]:
#short analysis of data above: All means are essentially zero (0.01%–0.04% per day). Over long periods, daily returns fluctuate around zero. 
#The Nikkei 225 is the most volatile, and the FTSE 100 is the least. 

In [ ]:
# 4. EDA: Prices, Returns, Rolling Volatility


# 4.1 Price series
price_data.plot(subplots=True, layout=(2, 2), figsize=(15, 8), 
                color='steelblue', title='Adjusted Close Prices')
plt.tight_layout()
plt.show()

# 4.2 Log returns series 
returns.plot(subplots=True, layout=(2, 2), figsize=(15, 8), 
             color='darkgreen', title='Daily Log Returns (%)')
plt.tight_layout()
plt.show()

# 4.3 30-day rolling volatility
rolling_vol = returns.rolling(window=30).std()
rolling_vol.plot(subplots=True, layout=(2, 2), figsize=(15, 8), 
                 color='crimson', title='30-Day Rolling Volatility (%)')
plt.tight_layout()
plt.show()

In [ ]:
# obervations for the three graphs above: 
#1. non-stationary price, 2008 crisis, 2020(2022) covid ... 
#2 Overall symmetricalups and downs, leveraging. Take time to go from very volatile periods (2008) back to normal. 


In [ ]:
# 4.5. Price Performance Together 

# All start at 100
normalized_prices = (price_data / price_data.iloc[0]) * 100

# All four on a single chart
plt.figure(figsize=(14, 7))
normalized_prices.plot(ax=plt.gca(), linewidth=1.5)

plt.title('Normalized Price Performance (Base = 100)', fontsize=14)
plt.ylabel('Indexed Value (Base 100)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend(title='Index', fontsize=11)
plt.grid(True, alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
#5. Return Distributions vs Normal + Kurtosis


fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, idx in enumerate(returns.columns):
    sns.histplot(returns[idx], kde=True, stat='density', bins=80,
                 ax=axes[i], color='steelblue', label='Actual', alpha=0.6)
    
    # Overlay Normal distribution with same mean/std
    mu, sigma = returns[idx].mean(), returns[idx].std()
    x = np.linspace(mu - 6*sigma, mu + 6*sigma, 300)
    axes[i].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label='Normal Fit')
    
    axes[i].set_title(f'{idx} — Return Distribution')
    axes[i].set_xlim(mu - 8*sigma, mu + 8*sigma)
    axes[i].legend()

plt.tight_layout()
plt.show()

# --- Quantify tail thickness ---
kurt = returns.kurtosis()          # Excess kurtosis (Normal = 0)
skew = returns.skew()              # Skewness (Normal = 0)

summary = pd.DataFrame({
    'Excess Kurtosis': kurt,
    'Skewness': skew,
    'Std Dev (%)': returns.std()
})

print("=== Distribution Shape Summary ===")
print("(Normal distribution: Excess Kurtosis = 0, Skewness = 0)\n")
display(summary.round(3))

In [ ]:
# interesting point: before doing this part I asume that because the Nikkei and EURO STOXX have higher standard deviations, they would naturally have the highest kurtosis
# maybe the actual reasoning should be: S&P 500 and FTSE 100 have tighter "normal day" distributions (lower standard deviations of 1.29% and 1.18%)
# but they still experienced massive absolute shocks (10%+), mathematically further out in standard deviations (8σ to 9σ), resulting in a higher excess kurtosis


In [ ]:
# 6. Fit GARCH(1,1) - Normal Distribution

garch_normal_results = {}

for idx in returns.columns:
    print(f"Fitting GARCH(1,1) with Normal errors for {idx}...")
    
    # Define the model:
    # vol='Garch' -> GARCH model
    # p=1, q=1    -> GARCH(1,1)
    # dist='normal' -> Normal distribution for errors
    model = arch_model(returns[idx], vol='Garch', p=1, q=1, dist='normal')
    
    # Fit the model, MLE
    # disp='off' only show final result
    # maxiter=1000 secrue termination mature
    res = model.fit(disp='off', update_freq=0, options={'maxiter': 1000})
    
    # Store the result object
    garch_normal_results[idx] = res
    
    print(f"{idx} fit complete. AIC = {res.aic:.2f}, BIC = {res.bic:.2f}\n")

In [ ]:
# All four AIC values are massive, fat tails significant

In [ ]:
# 7. Extract Parameters & Compute Persistence


params_df = pd.DataFrame(columns=['mu', 'omega', 'alpha', 'beta', 'persistence', 'AIC', 'BIC'])

for idx, res in garch_normal_results.items():
    params = res.params
    mu = params.get('mu', 0)
    omega = params['omega']
    alpha = params['alpha[1]']
    beta = params['beta[1]']
    persistence = alpha + beta
    params_df.loc[idx] = [mu, omega, alpha, beta, persistence, res.aic, res.bic]

print("=== GARCH(1,1)-Normal Parameter Estimates ===")
display(params_df.round(6))

In [ ]:
# 8. Plot Conditional Volatility & Persistence (plot cell 7 info)

# 8.1 Conditional Volatility (GARCH-estimated daily sigma)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (idx, res) in enumerate(garch_normal_results.items()):
    cond_vol = res.conditional_volatility
    axes[i].plot(cond_vol, color='purple', alpha=0.8, linewidth=0.8)
    axes[i].axvline(pd.Timestamp('2008-09-15'), color='red', linestyle='--', 
                    alpha=0.6, label='Lehman Crisis')
    axes[i].axvline(pd.Timestamp('2020-03-11'), color='orange', linestyle='--', 
                    alpha=0.6, label='COVID Crash')
    axes[i].set_title(f'{idx} — Conditional Volatility (GARCH)')
    axes[i].set_ylabel('Volatility (%)')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

# 8.2 Persistence Comparison Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
params_df['persistence'].sort_values(ascending=False).plot(
    kind='bar', color='teal', ax=ax, edgecolor='black'
)
ax.set_title('Volatility Persistence (α + β) Across Global Indices', fontsize=14)
ax.set_ylabel('Persistence (α + β)')
ax.set_xlabel('')
ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='Unit Root (Non-Stationary)')
ax.axhline(0.95, color='gray', linestyle=':', linewidth=1.5, label='0.95 Reference')
ax.set_ylim(0.94, 1.00)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 8.3 Alpha vs Beta comparison 
fig, ax = plt.subplots(figsize=(10, 5))
params_df[['alpha', 'beta']].plot(kind='bar', ax=ax, color=['steelblue', 'salmon'], edgecolor='black')
ax.set_title('Alpha (Shock Reaction) vs Beta (Memory) by Index', fontsize=14)
ax.set_ylabel('Parameter Value')
ax.set_xlabel('')
ax.legend(['Alpha (α)', 'Beta (β)'], fontsize=11)
ax.grid(axis='y', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 9. Diagnostics: Standardized Residuals & LB Test

for idx, res in garch_normal_results.items():
    # Standardized residuals = residuals / conditional volatility
    std_resid = res.resid / res.conditional_volatility
    
    # Create 4-panel diagnostic plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f'{idx} — Model Diagnostics', fontsize=16)
    
    # 1. Time series of standardized residuals 
    axes[0, 0].plot(std_resid, color='blue', alpha=0.6, linewidth=0.7)
    axes[0, 0].axhline(0, color='black', linestyle='--', linewidth=0.8)
    axes[0, 0].set_title('Standardized Residuals')
    
    # 2. Histogram vs Standard Normal (check for remaining non-normality)
    axes[0, 1].hist(std_resid, bins=60, density=True, alpha=0.7, color='steelblue', label='Actual')
    x = np.linspace(-4, 4, 200)
    axes[0, 1].plot(x, stats.norm.pdf(x, 0, 1), 'r-', lw=2, label='Standard Normal')
    axes[0, 1].set_title('Histogram vs Normal')
    axes[0, 1].legend()
    
    # 3. ACF of squared standardized residuals (should show no significant spikes)
    plot_acf(std_resid**2, ax=axes[1, 0], lags=20, title='ACF of Squared Residuals')
    
    # 4. Ljung-Box p-values on squared residuals (lags 1-10)
    lb_test = acorr_ljungbox(std_resid**2, lags=10, return_df=True)
    p_values = lb_test['lb_pvalue'].values
    axes[1, 1].bar(range(1, 11), p_values, color='green', edgecolor='black')
    axes[1, 1].axhline(0.05, color='red', linestyle='--', label='Significance (0.05)')
    axes[1, 1].set_title('Ljung-Box P-values (Squared Residuals)')
    axes[1, 1].set_xlabel('Lag')
    axes[1, 1].set_ylabel('P-value')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print diagnostic conclusion
    print(f"\n{idx} Ljung-Box test (squared residuals, lags 1-10):")
    print(f"  P-values: {np.round(p_values, 4)}")
    if all(p > 0.05 for p in p_values):
        print(" All p-values > 0.05. Cannot reject H0 of no autocorrelation. Model is adequate.\n")
    else:
        print(" Some p-values < 0.05. Remaining ARCH effects detected. Consider higher-order GARCH.\n")

In [ ]:
# 10.  Student-t Distribution

garch_t_results = {}

for idx in returns.columns:
    print(f"Fitting GARCH(1,1) with Student-t errors for {idx}...")
    
    # dist from 'normal' to 't'
    model_t = arch_model(returns[idx], vol='Garch', p=1, q=1, dist='t')
    res_t = model_t.fit(disp='off', update_freq=0, options={'maxiter': 1000})
    
    garch_t_results[idx] = res_t
    
    # degrees of freedom (nu)
    nu = res_t.params['nu']
    print(f"{idx} fit complete. DoF (ν) = {nu:.2f}, AIC = {res_t.aic:.2f}, BIC = {res_t.bic:.2f}\n")

In [ ]:
# 11. Compare Normal vs Student-t Models

# --- 11.1 Comparison DataFrame
comparison_df = pd.DataFrame(columns=['AIC_Normal', 'AIC_t', 'DoF_nu'])

for idx in returns.columns:
    aic_n = garch_normal_results[idx].aic
    aic_t = garch_t_results[idx].aic
    nu = garch_t_results[idx].params['nu']
    comparison_df.loc[idx] = [aic_n, aic_t, nu]

comparison_df['AIC_Improvement'] = comparison_df['AIC_Normal'] - comparison_df['AIC_t']
comparison_df['Kurtosis_Implied'] = 6 / (comparison_df['DoF_nu'] - 4) + 3  # For nu > 4

print("=== Normal vs Student-t Model Comparison ===")
display(comparison_df.round(2))

# --- 11.2 Plot AIC Comparison
fig, ax = plt.subplots(figsize=(10, 5))
comparison_df[['AIC_Normal', 'AIC_t']].plot(
    kind='bar', ax=ax, color=['salmon', 'steelblue'], edgecolor='black'
)
ax.set_title('AIC Comparison: Normal vs Student-t GARCH', fontsize=14)
ax.set_ylabel('AIC (Lower is Better)')
ax.set_xlabel('')
ax.legend(['Normal', 'Student-t'])
ax.grid(axis='y', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# --- 11.3 Plot Degrees of Freedom (Tail Thickness)
fig, ax = plt.subplots(figsize=(10, 5))
comparison_df['DoF_nu'].sort_values().plot(kind='bar', color='purple', ax=ax, edgecolor='black')
ax.set_title('Estimated Degrees of Freedom (ν) by Index', fontsize=14)
ax.set_ylabel('ν (Lower = Fatter Tails)')
ax.set_xlabel('')
ax.legend()
ax.grid(axis='y', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# --- 11.4 95 quantile
print("\n=== 95% VaR Quantile Comparison (Normal vs Student-t) ===")

for idx in comparison_df.index:
    nu = comparison_df.loc[idx, 'DoF_nu']
    # 5% left-tail quantiles
    norm_q = stats.norm.ppf(0.05)       # ~ -1.645
    t_q = stats.t.ppf(0.05, nu)         # Varies based on nu
    risk_ratio = abs(t_q / norm_q)
    print(f"{idx}: ν = {nu:.2f} | Normal Quantile = {norm_q:.3f} | t-Quantile = {t_q:.3f} | Risk Underestimated by {risk_ratio:.2f}x")